In [1]:
import pandas as pd

# Carregar os dados
df = pd.read_csv('customer_support_tickets.csv')

# Converter colunas de tempo
df['First Response Time'] = pd.to_datetime(df['First Response Time'])
df['Time to Resolution'] = pd.to_datetime(df['Time to Resolution'])

# Remover tickets sem dados de resolução e filtrar datas inconsistentes
df_resolved = df.dropna(subset=['First Response Time', 'Time to Resolution']).copy()
df_resolved['Duration_Hours'] = (df_resolved['Time to Resolution'] - df_resolved['First Response Time']).dt.total_seconds() / 3600
df_valid = df_resolved[df_resolved['Duration_Hours'] > 0].copy()

# Definições de custos
SLA_HOURS = 11
AGENT_HOURLY_RATE = 15
AI_TICKET_TYPES = ['Technical issue', 'Product inquiry']
AI_COST_PER_TICKET = 0.05  # Custo médio de tokens em reais

# 1. Cálculo do gasto extra (apenas horas que passaram do SLA)
df_valid['Extra_Hours'] = (df_valid['Duration_Hours'] - SLA_HOURS).clip(lower=0)
total_extra_cost = (df_valid['Extra_Hours'] * AGENT_HOURLY_RATE).sum()

# 2. Análise de Economia com IA
# Filtrar tickets que a IA resolveria
ai_tickets = df_valid[df_valid['Ticket Type'].isin(AI_TICKET_TYPES)].copy()

human_cost_ai_eligible = (ai_tickets['Duration_Hours'] * AGENT_HOURLY_RATE).sum()
total_ai_cost = len(ai_tickets) * AI_COST_PER_TICKET
potential_savings = human_cost_ai_eligible - total_ai_cost

print(f"Gasto Extra (SLA > 11h): R$ {total_extra_cost:.2f}")
print(f"Economia total com IA: R$ {potential_savings:.2f}")

Gasto Extra (SLA > 11h): R$ 24660.25
Economia total com IA: R$ 63258.95
